# Day 1 · Session 4 — Multimodal AI

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ksuresh/fdp-llm-agentic-ai/blob/main/day1-llm-foundations/04_multimodal_ai.ipynb)

---
**Duration:** ~30 minutes  
**Goal:** Send images, PDFs, and audio to LLMs. Build a multimodal teaching tool.

> *Adapted from Ed Donner's LLM Engineering course (week2). Simplified and contextualised for faculty.*

In [ ]:
!pip install openai anthropic google-generativeai pypdf python-dotenv -q
# For local Whisper: pip install openai-whisper

In [ ]:
import os, base64, json
from pathlib import Path
from openai import OpenAI
import anthropic
import google.generativeai as genai
from dotenv import load_dotenv

load_dotenv()

openai_client   = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
claude_client   = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))
gemini          = genai.GenerativeModel("gemini-2.0-flash")

print("✅ All clients ready")

---
## Part 1 — Vision: Describing Images

In [ ]:
# ── Method 1: Image from URL ─────────────────────────────────
# Use any publicly accessible image URL
# Replace with a diagram/circuit/chart relevant to your subject
IMAGE_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/1/14/Gatto_europeo4.jpg/320px-Gatto_europeo4.jpg"

response = openai_client.chat.completions.create(
    model="gpt-4o",
    messages=[{
        "role": "user",
        "content": [
            {"type": "image_url", "image_url": {"url": IMAGE_URL}},
            {"type": "text", "text": "Describe this image in detail. Then write a one-sentence caption suitable for a textbook."}
        ]
    }],
    max_tokens=400,
)

print("🖼️ Image Analysis:")
print(response.choices[0].message.content)

In [ ]:
# ── Method 2: Image from local file (base64 encoded) ─────────
def encode_image(image_path):
    """Convert local image file to base64 for API submission"""
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

def analyze_image_file(image_path, question, model="gpt-4o"):
    """Analyze a local image file"""
    b64_image = encode_image(image_path)
    ext = Path(image_path).suffix.lower()
    media_type = {'.jpg':"image/jpeg", '.jpeg':"image/jpeg",
                  '.png':"image/png", '.gif':"image/gif",
                  '.webp':"image/webp"}.get(ext, "image/jpeg")

    response = openai_client.chat.completions.create(
        model=model,
        messages=[{
            "role": "user",
            "content": [
                {"type": "image_url",
                 "image_url": {"url": f"data:{media_type};base64,{b64_image}"}},
                {"type": "text", "text": question}
            ]
        }],
        max_tokens=500,
    )
    return response.choices[0].message.content

print("Function ready. Usage:")
print('  result = analyze_image_file("path/to/image.jpg", "What is in this image?")')

---
## Part 2 — Vision: Educational Use Cases

In [ ]:
# ── Auto-grade handwritten work ───────────────────────────────
# Replace IMAGE_URL with a photo of handwritten student work
HANDWRITING_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/2/21/Simple_algebra_example.jpg/320px-Simple_algebra_example.jpg"

grade_prompt = """
You are grading a handwritten student answer on algebra.

Task: Grade this student's work step by step.
For each step, identify if it is:
  ✓ Correct
  ✗ Incorrect (explain the error)
  ? Unclear (explain what's unclear)

Provide:
1. Step-by-step grading
2. Final marks out of 10
3. One encouraging comment
4. One specific improvement suggestion
"""

response = openai_client.chat.completions.create(
    model="gpt-4o",
    messages=[{
        "role": "user",
        "content": [
            {"type": "image_url", "image_url": {"url": HANDWRITING_URL}},
            {"type": "text", "text": grade_prompt}
        ]
    }],
    max_tokens=600,
)

print("📝 Handwritten Work Grading:")
print(response.choices[0].message.content)

In [ ]:
# ── Accessibility: generate alt-text for diagrams ─────────────
# A simple diagram URL — replace with any technical diagram
DIAGRAM_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/PNG_transparency_demonstration_1.png/280px-PNG_transparency_demonstration_1.png"

alt_text = openai_client.chat.completions.create(
    model="gpt-4o",
    messages=[{
        "role": "user",
        "content": [
            {"type": "image_url", "image_url": {"url": DIAGRAM_URL}},
            {"type": "text", "text": """
Generate detailed alt-text for this image that would help a visually impaired
engineering student understand its content fully. Include:
- What type of image/diagram it is
- All labels, annotations, and key elements
- The relationships between elements
- Any trends, patterns, or conclusions visible
Keep it under 150 words but make it comprehensive.
            """}
        ]
    }],
    max_tokens=300,
)

print("♿ Generated Alt-Text:")
print(alt_text.choices[0].message.content)

---
## Part 3 — Documents: PDF Understanding

In [ ]:
# ── Approach 1: Extract text with pypdf, then query ───────────
import pypdf

def query_pdf_text(pdf_path, question, max_chars=15000):
    """Extract PDF text and query it with GPT-4o-mini"""
    reader = pypdf.PdfReader(pdf_path)
    text = "\n\n".join([
        f"--- Page {i+1} ---\n{page.extract_text()}"
        for i, page in enumerate(reader.pages)
    ])
    # Truncate if too long
    text = text[:max_chars]

    print(f"📄 PDF loaded: {len(reader.pages)} pages, {len(text)} characters")

    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You answer questions based ONLY on the provided document. If the answer is not in the document, say so clearly."},
            {"role": "user", "content": f"Document:\n{text}\n\nQuestion: {question}"}
        ],
        max_tokens=500,
    )
    return response.choices[0].message.content

# Usage — replace with your own PDF path:
# answer = query_pdf_text("data/sample_syllabus.pdf", "What are the prerequisites for this course?")
# print(answer)

# Demo with any text file treated as content:
demo_text = """
Course: CS601 Machine Learning
Credits: 4
Prerequisites: Linear Algebra, Probability Theory, Python Programming
Instructor: Dr. Suresh Kumar
Schedule: Monday, Wednesday, Friday 10:00 AM - 11:00 AM

Unit 1: Introduction to ML (Weeks 1-2)
  - Types of ML: Supervised, Unsupervised, Reinforcement
  - Bias-Variance Tradeoff
  - Model Evaluation Metrics

Unit 2: Regression (Weeks 3-4)
  - Linear Regression
  - Polynomial Regression
  - Regularisation: L1 and L2
"""

response_demo = openai_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Answer questions based only on the provided course syllabus."},
        {"role": "user", "content": f"Syllabus:\n{demo_text}\n\nWhat are the prerequisites and when are classes held?"}
    ],
    max_tokens=200,
)
print("📄 PDF Query Result (demo):")
print(response_demo.choices[0].message.content)

In [ ]:
# ── Approach 2: Send PDF directly to Claude (best for layout) ─
def query_pdf_claude(pdf_path, question):
    """Send PDF natively to Claude — preserves tables, layout, figures"""
    with open(pdf_path, "rb") as f:
        pdf_data = base64.standard_b64encode(f.read()).decode("utf-8")

    message = claude_client.messages.create(
        model="claude-opus-4-6",
        max_tokens=1000,
        messages=[{
            "role": "user",
            "content": [
                {
                    "type": "document",
                    "source": {
                        "type": "base64",
                        "media_type": "application/pdf",
                        "data": pdf_data
                    }
                },
                {"type": "text", "text": question}
            ]
        }]
    )
    return message.content[0].text

# Usage:
# result = query_pdf_claude("report.pdf", "Summarise the key findings in 5 bullet points")

print("✅ Claude PDF function ready")
print("Usage: result = query_pdf_claude('your_file.pdf', 'Your question here')")

In [ ]:
# ── Approach 3: Gemini native PDF (free tier, 1M context) ─────
def query_pdf_gemini(pdf_path, question):
    """Use Gemini's native PDF support — great for very long documents"""
    uploaded = genai.upload_file(pdf_path)
    response = gemini.generate_content([question, uploaded])
    return response.text

# Usage:
# result = query_pdf_gemini("thesis.pdf", "Extract all tables as markdown")

print("✅ Gemini PDF function ready")
print("Usage: result = query_pdf_gemini('thesis.pdf', 'Extract all tables as markdown')")
print("\nGemini's advantage: 1 million token context = can process ~3,000 pages in one call")

---
## Part 4 — Audio: Speech-to-Text with Whisper

In [ ]:
# ── Whisper via OpenAI API ────────────────────────────────────
def transcribe_audio(audio_path, language=None):
    """
    Transcribe audio file using Whisper.
    Supports: mp3, mp4, mpeg, mpga, m4a, wav, webm
    language: 'en', 'hi', 'ta', 'te', 'kn', 'mr' etc. (None = auto-detect)
    """
    with open(audio_path, "rb") as audio_file:
        kwargs = dict(
            model="whisper-1",
            file=audio_file,
            response_format="verbose_json",  # includes word-level timestamps
        )
        if language:
            kwargs["language"] = language

        transcript = openai_client.audio.transcriptions.create(**kwargs)

    return {
        "text": transcript.text,
        "language": transcript.language,
        "duration": transcript.duration,
    }

# Usage:
# result = transcribe_audio("student_question.mp3")
# result = transcribe_audio("lecture_hindi.mp3", language="hi")

print("✅ Whisper transcription function ready")
print("\nSupported Indian languages:")
langs = {"hi":"Hindi", "ta":"Tamil", "te":"Telugu", "kn":"Kannada",
         "mr":"Marathi", "gu":"Gujarati", "bn":"Bengali", "ml":"Malayalam"}
for code, name in langs.items():
    print(f"  language='{code}' → {name}")

In [ ]:
# ── Local Whisper (completely free, no API key) ───────────────
# First: pip install openai-whisper
# Then: runs entirely on your machine

WHISPER_DEMO_CODE = '''
import whisper

# Models: tiny, base, small, medium, large
# Larger = more accurate but slower
model = whisper.load_model("base")  # ~150MB download

# Auto-detect language:
result = model.transcribe("audio.mp3")
print(result["text"])
print(f"Detected language: {result['language']}")

# Force Tamil:
result_ta = model.transcribe("lecture_tamil.mp3", language="ta")
print(result_ta["text"])
'''
print("Local Whisper code (run after installing openai-whisper):")
print(WHISPER_DEMO_CODE)

---
## Part 5 — Fusion: Combining Modalities

In [ ]:
# ── Send image + text in one call ────────────────────────────
# This is the most powerful pattern: text + image + optional document

# Example: Engineering diagram analysis with context
CIRCUIT_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/1/11/Ohm%27s_Law_with_Voltage_source_TeX.svg/240px-Ohm%27s_Law_with_Voltage_source_TeX.svg.png"

fusion_response = openai_client.chat.completions.create(
    model="gpt-4o",
    messages=[{
        "role": "system",
        "content": "You are a teaching assistant for a 1st year B.Tech electrical engineering course."
    }, {
        "role": "user",
        "content": [
            {"type": "image_url", "image_url": {"url": CIRCUIT_URL}},
            {"type": "text", "text": """
Analyse this circuit diagram for a student who has just learned Ohm's Law.

Please provide:
1. What components are in the circuit?
2. Describe the relationship shown (V = IR)
3. One exam question a professor might ask about this diagram
4. Common mistakes students make with this concept
            """}
        ]
    }],
    max_tokens=500,
)

print("🔗 Fusion Analysis (Image + Text + System Context):")
print(fusion_response.choices[0].message.content)

In [ ]:
# ── Full multimodal teaching assistant ────────────────────────
# Combines: system persona + image + structured output

def multimodal_teaching_assistant(image_url, student_question, subject, level):
    """
    A complete multimodal teaching assistant that:
    1. Understands the image context
    2. Answers the student's question
    3. Returns structured educational content
    """
    response = openai_client.chat.completions.create(
        model="gpt-4o",
        messages=[{
            "role": "system",
            "content": f"""
You are a teaching assistant for {subject} at {level} level in an Indian engineering college.
When given an image and a student question:
1. First understand what the image shows
2. Answer the student's specific question
3. Connect the image to the course concept
4. End with a Socratic follow-up question

Use Indian examples and keep language accessible.
            """
        }, {
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": image_url}},
                {"type": "text",      "text": f"Student question: {student_question}"}
            ]
        }],
        max_tokens=600,
    )
    return response.choices[0].message.content

# Test it:
result = multimodal_teaching_assistant(
    image_url=CIRCUIT_URL,
    student_question="I don't understand how increasing resistance affects current. Can you show me using this diagram?",
    subject="Basic Electrical Engineering",
    level="1st year B.Tech"
)
print("🎓 Multimodal Teaching Assistant Response:")
print(result)

---
## 🎯 Lab Assignment

**File:** `lab/lab4_multimodal.ipynb`

### Task 1 — Vision
Find an image relevant to your subject (diagram, chart, circuit, microscopy, etc.).
Send it to GPT-4o and ask it to explain the content for your students.

### Task 2 — PDF
Take a PDF from your subject (your own notes, a textbook chapter, or the sample syllabus in `data/`).
Extract and summarise 3 key points using the text extraction method.

### Task 3 — Audio
Record a 30-second voice explanation of a concept in your subject language.
Transcribe it with Whisper. Check accuracy.

### Task 4 — Fusion
Find an image + write a student question about it.
Use `multimodal_teaching_assistant()` with your subject and level.

### 🌟 Bonus — Smart Grader
Build a function `smart_grader(image_url, rubric)` that:
- Accepts a photo of handwritten student work
- Applies your rubric
- Returns structured JSON with marks per criterion and feedback

```python
# Your code here
```